# 🚀 Классический пример: DDPG для F‑16 (LinearLongitudinalF16-v0)

Лаконичный, удобный для чтения на GitHub пример обучения DDPG на линейной продольной модели F‑16: настройки, инициализация, запуск обучения и быстрая визуализация прогресса.


## 📋 Оглавление
- [📖 Введение](#📖-Введение)
- [📦 Импорты](#📦-Импорты)
- [🔧 Параметры](#🔧-Параметры)
- [✈️ Среда](#✈️-Среда)
- [🤖 Обучение DDPG](#🤖-Обучение-DDPG)
- [📈 Визуализация](#📈-Визуализация)
- [💡 Советы](#💡-Советы)
- [📄 Лицензия](#📄-Лицензия)


## 📖 Введение
DDPG — off-policy алгоритм с детерминированной политикой для непрерывных действий. Здесь обучаем агента стабилизировать маятник в среде Pendulum-v1.


In [ ]:
# 📦 Импорты
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt

from tensoraerospace.agent.ddpg.model import DDPG
from tensoraerospace.utils import generate_time_period, convert_tp_to_sec_tp
from tensoraerospace.signals.standart import unit_step

# 🔧 Параметры
seed = 42
np.random.seed(seed)

replay_size = 1_000_000
actor_lr = 1e-3
critic_lr = 1e-4
train_frames = 12000
print_every = 500
batch_size = 128

# Временная сетка и референс-сигнал (как в GAIL)
dt = 0.01
tp = generate_time_period(tn=20, dt=dt)
tps = convert_tp_to_sec_tp(tp, dt=dt)
number_time_steps = len(tp)
reference_signals = np.reshape(
    unit_step(degree=5, tp=tp, time_step=1000, output_rad=True), [1, -1]
)

# ✈️ Среда LinearLongitudinalF16-v0 c обёртками под DDPG
class FlattenObs(gym.ObservationWrapper):
    def __init__(self, env):
        super().__init__(env)
        original = env.observation_space
        self.observation_space = gym.spaces.Box(
            low=-np.inf, high=np.inf, shape=(original.shape[0],), dtype=np.float32
        )

    def observation(self, observation):
        return observation.reshape(-1).astype(np.float32)

class NormalizedActions(gym.ActionWrapper):
    def action(self, action):
        low_bound = self.action_space.low
        upper_bound = self.action_space.high
        action = low_bound + (action + 1.0) * 0.5 * (upper_bound - low_bound)
        return np.clip(action, low_bound, upper_bound)

    def reverse_action(self, action):
        low_bound = self.action_space.low
        upper_bound = self.action_space.high
        action = 2 * (action - low_bound) / (upper_bound - low_bound) - 1
        return np.clip(action, low_bound, upper_bound)

# Создаём базовую среду и применяем обёртки
env_base = gym.make(
    'LinearLongitudinalF16-v0',
    number_time_steps=number_time_steps,
    initial_state=[[0], [0], [0]],  # theta, alpha, q
    reference_signal=reference_signals,
    use_reward=True,
    state_space=["theta", "alpha", "q"],
    output_space=["theta", "alpha", "q"],
    control_space=["ele"],
    tracking_states=["alpha"],
)

env = NormalizedActions(FlattenObs(env_base))

# 🤖 Обучение DDPG
agent = DDPG(env, actor_lr, critic_lr, replay_size)
agent.learn(train_frames, print_every, batch_size)

2024-07-30 08:20:06.617740: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-07-30 08:20:06.617773: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-07-30 08:20:06.618389: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-07-30 08:20:06.622701: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-07-30 08:20:07.254391: W tensorflow/compiler/tf2

-3500.8337473891365
-3589.4876147512236
-2197.0131623913207
-1858.0237090207388
-1272.014623989684
-1212.7939297872063
-1197.3503752233062
-753.8523564655063
-1057.0673585256325
-774.4367515912473
-628.3070771533033
-509.580943392905
-506.72415759831176
-1018.2413158148908
-890.5926284650565
-744.808548877132
-1131.9461972053632
-1019.1800693146189
-803.4071011635544
-1140.3282973414496
-875.98839626264
-890.5032597943205
-1014.317796226746
-1014.6442220416678


In [ ]:
# 📈 Визуализация
# Быстрый график суммарной награды по эпизодам, если агент её сохраняет
if hasattr(agent, 'rewards') and isinstance(agent.rewards, list) and len(agent.rewards) > 0:
    plt.figure(figsize=(8,4))
    plt.plot(agent.rewards)
    plt.xlabel('Episode')
    plt.ylabel('Reward')
    plt.title('DDPG training progress (episode reward)')
    plt.grid(True, alpha=0.3)
else:
    print('No reward history available from agent.')


## 💡 Советы
- Зафиксируйте сиды NumPy/PyTorch для повторяемости
- Уменьшайте `actor_lr`/`critic_lr`, если наблюдаете нестабильность
- Увеличьте `replay_size` и `batch_size` для более устойчивого обучения

## 📄 Лицензия
Этот пример распространяется на условиях лицензии проекта (см. LICENSE в корне репозитория).
